# fractional-stride-zero-insertion — worked example 3: Zero-Inserted Output Size: Predicting Shape Before Allocation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `fractional-stride-zero-insertion`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Before allocating the zero-inserted tensor, you should compute its output size using the formula `(L - 1) * s + 1` for each spatial dimension. This formula works for any stride `s ≥ 1`: when `s = 1` it returns `L` (unchanged), and for larger strides it inserts the necessary zeros to space the original elements `s` positions apart.

## Worked solution

We verify the output size formula for several `(L, s)` combinations and then confirm the allocated tensor has the right shape.

**Formula derivation:** The last original element sits at index `(L-1) * s` (zero-indexed) in the output. Since indices are 0-based, the tensor length is `(L-1)*s + 1`.

**Examples:**
- `L=1, s=4`: output length = `(1-1)*4 + 1 = 1`. A single-element input always gives a single-element output regardless of stride.
- `L=5, s=2`: `(5-1)*2 + 1 = 9`. Four gaps, each filled with one zero.
- `L=3, s=3`: `(3-1)*3 + 1 = 7`. Two gaps, each filled with two zeros.

**2-D:** Apply the formula independently to H and W. For input `(B, C, H, W)` and stride `s`, output is `(B, C, (H-1)*s+1, (W-1)*s+1)`.

In [ ]:
import torch as t

def predict_zero_insert_shape(B, C, H, W, s):
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    return (B, C, H_out, W_out)

def zero_insert_2d(x: t.Tensor, s: int) -> t.Tensor:
    B, C, H, W = x.shape
    out_shape = predict_zero_insert_shape(B, C, H, W, s)
    y = t.zeros(*out_shape, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y

# Verify the formula on a variety of inputs
cases = [
    (1, 1, 1, 1, 4),   # single pixel, stride 4
    (2, 3, 5, 5, 2),   # 5x5 input, stride 2
    (1, 1, 3, 4, 3),   # rectangular input, stride 3
    (1, 1, 7, 7, 1),   # stride 1 -> no change
]

for B, C, H, W, s in cases:
    expected = predict_zero_insert_shape(B, C, H, W, s)
    x = t.randn(B, C, H, W)
    y = zero_insert_2d(x, s)
    assert y.shape == t.Size(expected), f"{y.shape} != {expected}"
    print(f"({B},{C},{H},{W}) stride={s} -> predicted {expected} actual {tuple(y.shape)} ✓")

print("All shape predictions verified.")